# JarvisLM-350M: Colab A100 smoke training

目标：在 **保持 353.5M 模型架构不变** 的前提下，验证 A100 能完成 FineWeb-Edu 数据准备、bf16 前反向传播、Muon + AdamW 更新与 checkpoint 保存。

成功条件：GPU 是 A100；训练完成 10 个 update；loss、grad norm 与 tokens/s 均为有限数；产生 `last.pt`。这不是 10B-token 正式训练，不能据此报告最终 PPL、吞吐或 MFU。

In [ ]:
# Colab runtime: Runtime -> Change runtime type -> A100 GPU
import torch

assert torch.cuda.is_available(), '请在 Colab 中选择 GPU runtime。'
gpu_name = torch.cuda.get_device_name(0)
assert 'A100' in gpu_name.upper(), f'此 notebook 预期 A100，当前是: {gpu_name}'
print({'gpu': gpu_name, 'memory_gb': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'bf16': torch.cuda.is_bf16_supported()})

### 安全加载 token（IDE 模式）

IDE 连接的 Colab 没有侧边栏 Secrets。此 cell 用隐藏输入临时写入当前 Jupyter kernel 的环境变量；不会显示或保存 token。运行时重启后需要重新输入。

In [ ]:
import os
from getpass import getpass

def prompt_secret(name: str, required: bool = False) -> bool:
    if os.environ.get(name):
        return True
    prompt = f'{name} (hidden input' + (', required' if required else ', optional') + '): '
    value = getpass(prompt)
    if value:
        os.environ[name] = value
        return True
    if required:
        raise RuntimeError(f'{name} is required for this run')
    return False

hf_token_loaded = prompt_secret('HF_TOKEN')
wandb_api_key_loaded = prompt_secret('WANDB_API_KEY')
print({'hf_token_loaded': hf_token_loaded, 'wandb_api_key_loaded': wandb_api_key_loaded})

## Colab Secrets 与安装项目

先在 Colab 左侧 **Secrets** 添加 `HF_TOKEN`；若要启用 W&B，再添加 `WANDB_API_KEY`。不要把 token 写进 notebook。下面的安装 cell 会拉取 `codex/jarvislm-350m-training` 分支，并可重复运行。

In [ ]:
%cd /content
!if [ -d small-llm-from-scratch/.git ]; then git -C small-llm-from-scratch fetch origin codex/jarvislm-350m-training && git -C small-llm-from-scratch checkout codex/jarvislm-350m-training && git -C small-llm-from-scratch pull --ff-only; else git clone --branch codex/jarvislm-350m-training https://github.com/JarvisZhang24/small-llm-from-scratch.git; fi
%cd /content/small-llm-from-scratch
!python -m pip install -q -e '.[dev]'
!PYTHONPATH=src python -c "import torch, jarvislm; print('torch:', torch.__version__)"

## 固定本次 smoke 配置

模型仍是 24 层、1024 hidden、1024 context 的 350M 基线。仅为了适配 Colab 时长与显存，测试改为 micro-batch=2、梯度累积=8、10 updates，并先关闭 `torch.compile`。正式 H100 配置仍是 16 × 32。

In [ ]:
from pathlib import Path

RUN_ROOT = Path('/content/jarvislm-a100-smoke')
TRAIN_DIR = RUN_ROOT / 'data/train'
VAL_DIR = RUN_ROOT / 'data/val'
CHECKPOINT_DIR = RUN_ROOT / 'checkpoints'
TRAIN_TOKENS = 1_000_000
VAL_TOKENS = 100_000
MAX_STEPS = 10
MICRO_BATCH = 2
GRAD_ACCUM = 8
print({'tokens_per_update': 1024 * MICRO_BATCH * GRAD_ACCUM, 'run_root': str(RUN_ROOT)})

## 准备小规模、互不重叠的 FineWeb-Edu shards

首次运行会下载并 tokenize 110 万 token，通常需要几分钟。若目录已经存在，训练器会拒绝覆盖；需要重来时请换 `RUN_ROOT`，不要在 notebook 中静默删除数据。

In [ ]:
!PYTHONPATH=src python -m jarvislm.training.train --prepare-data --prepare-only --data-dir {TRAIN_DIR} --val-dir {VAL_DIR} --prepare-train-tokens {TRAIN_TOKENS} --prepare-val-tokens {VAL_TOKENS}

## A100 350M smoke training

这里显式使用 `--no-require-h100`，它只放宽硬件检查，不会改变 350M 模型结构。先使用 `--no-compile` 降低第一次排错难度；这步成功后再进行 compile 检查。W&B 默认为关闭，避免训练验证被账户配置阻塞。

In [ ]:
!PYTHONPATH=src python -m jarvislm.training.train --data-dir {TRAIN_DIR} --val-dir {VAL_DIR} --checkpoint-dir {CHECKPOINT_DIR} --max-steps {MAX_STEPS} --micro-batch-size {MICRO_BATCH} --grad-accumulation-steps {GRAD_ACCUM} --num-workers 2 --no-require-h100 --no-compile --no-wandb --no-resume

In [ ]:
import torch

checkpoint = torch.load(CHECKPOINT_DIR / 'last.pt', map_location='cpu', weights_only=False)
summary = {
    'completed_step': checkpoint['step'],
    'parameter_tensors': len(checkpoint['model']),
    'has_muon': 'muon' in checkpoint,
    'has_ema': 'ema' in checkpoint,
}
assert summary['completed_step'] == MAX_STEPS
summary

## 可选：验证 `torch.compile`

确认上一阶段成功后，把下面 cell 的 `--no-compile` 删除，并改用新的 checkpoint 目录跑 2 steps。compile 首次编译会显著增加第一步时间，所以不要把它当作吞吐结果。

In [ ]:
# 可选：取消下一行开头的 # 后运行。
# !PYTHONPATH=src python -m jarvislm.training.train --data-dir {TRAIN_DIR} --val-dir {VAL_DIR} --checkpoint-dir {RUN_ROOT / 'compiled-checkpoints'} --max-steps 2 --micro-batch-size {MICRO_BATCH} --grad-accumulation-steps {GRAD_ACCUM} --num-workers 2 --no-require-h100 --compile --no-wandb --no-resume

## 记录结果与下一步

记录：GPU 型号、显存、每一步 loss/grad norm/tokens/s、是否保存 checkpoint。

若成功：下一阶段是在同一 A100 上增加到 100 steps，再迁移到 H100 的正式 10B-token 配置。若 OOM：只将 `MICRO_BATCH` 降到 1，并相应提高 `GRAD_ACCUM`；不要改动模型宽度、层数或 context。